> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../data/README.md).



# Capítulo 05 — Realce e Correção de Imagens

> 📖 **Referência:** Canty (2019), Capítulo 05

Notas em `aulas/realce_e_correcao.md`.

---

## 1. Intuição

Antes de classificar ou medir mudança, muitas vezes precisamos de duas coisas distintas:

1. **Realce** — tornar a imagem legível (estiramento de contraste, equalização de histograma).
2. **Correção** — ajustar radiometria (calibração, atmosfera) ou geometria (registro, GCPs).

Além disso, o capítulo trata **feições locais** (bordas Canny, cantos) usadas em registro e interpretação. Realce muda a escala dos valores para visualização; correção geométrica muda *onde* o pixel cai no mapa.

## 2. Fundamentos

### Contrast stretch (percentis)

$$
x' = \mathrm{clip}\left(\frac{x - x_{p_{\mathrm{low}}}}{x_{p_{\mathrm{high}}} - x_{p_{\mathrm{low}}}}, 0, 1\right)
$$

### Equalização de histograma

Usa a CDF empírica dos níveis de cinza para redistribuir intensidades de forma aproximadamente uniforme. Bom para visualização; perigoso se a análise for quantitativa entre cenas.

### Radiométrica vs geométrica

| | Radiométrica | Geométrica |
|---|--------------|------------|
| Mexe em | valor do pixel | posição no mapa |
| Exemplos | calibração, atmosfera, C-correction | georref, orto, RST com GCPs |

## 3. Implementação — realce

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
EXEMPLO = REPO / "data" / "exemplos" / "exemplo_sintetico.tif"
OUT = REPO / "conteudo/pre-processamento/figuras"
OUT.mkdir(parents=True, exist_ok=True)
BAND = 1


def read_band(path, band=1):
    try:
        import rasterio

        with rasterio.open(path) as ds:
            return ds.read(band).astype(np.float64)
    except Exception:
        pass
    try:
        from osgeo import gdal
        from osgeo.gdalconst import GA_ReadOnly

        gdal.AllRegister()
        ds = gdal.Open(str(path), GA_ReadOnly)
        arr = ds.GetRasterBand(band).ReadAsArray().astype(np.float64)
        ds = None
        return arr
    except Exception:
        pass
    import tifffile

    data = tifffile.imread(str(path))
    if data.ndim == 2:
        return data.astype(np.float64)
    if data.shape[0] < data.shape[-1]:
        return data[band - 1].astype(np.float64)
    return data[:, :, band - 1].astype(np.float64)


def contrast_stretch(img, p_low=2, p_high=98):
    lo, hi = np.percentile(img, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(img)
    return np.clip((img - lo) / (hi - lo), 0, 1)


def histogram_equalization(img):
    flat = img.ravel()
    vmin, vmax = float(flat.min()), float(flat.max())
    scaled = np.clip((flat - vmin) / (vmax - vmin) * 255.0, 0, 255).astype(np.uint8)
    hist, _ = np.histogram(scaled, bins=256, range=(0, 256))
    cdf = hist.cumsum().astype(np.float64)
    cdf_m = np.ma.masked_equal(cdf, 0)
    lut = ((cdf_m - cdf_m.min()) / (cdf_m.max() - cdf_m.min())).filled(0)
    return lut[scaled].reshape(img.shape)


img = read_band(EXEMPLO, BAND)
stretched = contrast_stretch(img)
equalized = histogram_equalization(img)
print(img.shape)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes[0, 0].imshow(contrast_stretch(img, 0, 100), cmap="gray")
axes[0, 0].set_title("Original (min–max)")
axes[0, 1].imshow(stretched, cmap="gray")
axes[0, 1].set_title("Stretch 2–98%")
axes[0, 2].imshow(equalized, cmap="gray")
axes[0, 2].set_title("Equalização")
for ax in axes[0]:
    ax.axis("off")

axes[1, 0].hist(img.ravel(), bins=64, color="gray", density=True)
axes[1, 0].set_title("Hist. original")
axes[1, 1].hist(stretched.ravel(), bins=64, color="steelblue", density=True)
axes[1, 1].set_title("Hist. stretch")
axes[1, 2].hist(equalized.ravel(), bins=64, color="darkorange", density=True)
axes[1, 2].set_title("Hist. equalizado")
fig.tight_layout()
fig.savefig(OUT / "nb05_realce.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("salvo", OUT / "nb05_realce.png")

## 4. Detecção de feições (cantos e Canny)

Equivalente visual ao `ex05_1.py` / `ex05_viz.py`, gravando só PNG em `figuras/`.

In [ ]:
import cv2 as cv

u8 = (stretched * 255).astype(np.uint8)
corners = cv.cornerMinEigenVal(u8, 7)
canny = cv.Canny(u8, 50, 150)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(stretched, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(contrast_stretch(corners), cmap="inferno")
axes[1].set_title("Cantos (min eigenvalue)")
axes[2].imshow(canny, cmap="gray")
axes[2].set_title("Canny")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "nb05_corner_canny.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("salvo", OUT / "nb05_corner_canny.png")

## 5. Scripts do livro e `legado/canty/auxil/`

| Script | Observação |
|--------|------------|
| `exercicios/ex05_1.py` | Escreve GeoTIFF `_corner` / `_canny` **ao lado do input** |
| `exercicios/ex05_2.py` | Contornos + Hu (usa `plt.show()`) |
| `exercicios/ex05_3.py` | RST a partir de arquivo de GCPs |
| `exercicios/ex05_4.py` | Stretch vs histeq → `figuras/` |
| `exercicios/ex05_viz.py` | PNGs versionáveis de cantos/Canny/Hu |

Correção topográfica no `legado/canty/auxil/`:

```bash
# %run ../../legado/canty/auxil/scripts/c_corr.py ...
```

## 6. Exercícios — comandos

```bash
python exercicios/ex05_4.py
python exercicios/ex05_viz.py

# originais do livro (GeoTIFF ao lado do input / janela interativa)
python exercicios/ex05_1.py -b 1 -a 1 caminho/exemplo_sintetico.tif   # cantos
python exercicios/ex05_1.py -b 1 -a 2 caminho/exemplo_sintetico.tif   # Canny
python exercicios/ex05_2.py -b 1 caminho/exemplo_sintetico.tif
python exercicios/ex05_3.py caminho/gcps.txt
```